# Модуль 1 · Бонус — Багатозадачність: `threading`, `multiprocessing`, `asyncio`

🎁 Це поглиблення теми **GIL / паралелізм** з [Уроку 4](lektsiya-4-funktsiyi-oblast-vydymosti-legb.ipynb).
Тема — улюблена на **співбесідах середнього рівня**, тож розберемо її **дуже детально** й так, як
ви б розповідали інтерв'юеру: спершу модель мислення, потім кожен інструмент, а в кінці — коли що обирати.

**Що всередині:** concurrency vs parallelism · GIL · `threading` (локи, семафори, черги, пули) ·
`multiprocessing` (процеси, IPC, shared memory) · **`asyncio` дуже детально** (event loop, таски,
`gather`, `Lock`, `Semaphore`, `Queue`, `run_in_executor`).

> 🧪 `threading` та `asyncio` — **запускні** прямо тут. `multiprocessing` — приклади коду
> (у Jupyter воно коректно не запускається через spawn/pickling — це пояснено в розділі 4).
> Метадані ноутбука: `asyncio`-клітинки використовують `await main()` (працює в Jupyter);
> у звичайному `.py` замість цього — `asyncio.run(main())`.

## 1. Головна модель: concurrency vs parallelism

Це **перше**, що варто чітко проговорити на співбесіді.

- **Concurrency (конкурентність)** — керування **багатьма задачами, що прогресують «разом»**
  через **перемикання** між ними. Це про **структуру** програми. Один кухар готує 3 страви,
  перемикаючись між ними, поки щось «вариться».
- **Parallelism (паралелізм)** — **буквально одночасне** виконання на **кількох ядрах** CPU.
  Три кухарі готують три страви **водночас**.

Concurrency ≠ parallelism: можна мати конкурентність без паралелізму (один потік, `asyncio`).

### Два типи задач (від цього залежить вибір інструмента)
- **I/O-bound** — програма більшість часу **чекає** (мережа, диск, БД). Тут виграє
  конкурентність: поки одна задача чекає, працює інша. → `asyncio` або `threading`.
- **CPU-bound** — програма більшість часу **рахує** (обробка зображень, математика). Тут потрібен
  **справжній паралелізм** на ядрах. → `multiprocessing`.

## 2. 🔎 GIL — чому потоки не пришвидшують обчислення

**GIL (Global Interpreter Lock)** — глобальне блокування в CPython, через яке **в один момент
часу байткод виконує лише ОДИН потік**, навіть якщо ядер багато.

**Навіщо він:** спрощує керування пам'яттю CPython (підрахунок посилань, refcounting) — без GIL
довелося б ставити блокування на кожен об'єкт, що складно й повільно.

**Наслідки (це і питають):**
- **CPU-bound + `threading` = НЕ швидше** (часто навіть повільніше — витрати на перемикання):
  потоки по черзі тримають GIL, реального паралелізму немає.
- **I/O-bound + `threading` = швидше:** коли потік чекає I/O, він **відпускає GIL**, і працює інший.
- Щоб паралелити **обчислення** — потрібні **процеси** (`multiprocessing`): у кожного свій
  інтерпретатор і **свій GIL**.

> 💡 GIL є у **CPython** (стандартному). У Jython/PyPy-STM його немає. У Python 3.13+ з'явився
> експериментальний режим **без GIL** (free-threading), але це поки що не дефолт.

## 3. `threading` — потоки (для I/O-bound)

Потоки живуть **в одному процесі** й **ділять пам'ять**. Створюємо `Thread`, `start()` запускає,
`join()` чекає завершення.

In [ ]:
import threading, time

def worker(name):
    print(f"  {name}: почав")
    time.sleep(0.3)                 # імітація очікування I/O (тут потік відпускає GIL)
    print(f"  {name}: завершив")

start = time.perf_counter()
threads = [threading.Thread(target=worker, args=(f"потік-{i}",)) for i in range(3)]
for t in threads: t.start()         # запускаємо всі
for t in threads: t.join()          # чекаємо всіх
print(f"3 потоки по 0.3с зайняли {time.perf_counter() - start:.2f}с (а не 0.9с — бо чекали ПАРАЛЕЛЬНО)")

### 🔎 Race condition і `Lock`

Оскільки потоки **ділять пам'ять**, вони можуть одночасно міняти одні дані — виникає **гонка
(race condition)**. Операція `counter += 1` **не атомарна** (читання → додавання → запис), і GIL
може перемкнути потік посередині.

In [ ]:
import threading

# ❌ БЕЗ блокування — гонка: результат менший за очікуваний
counter = 0
def increment():
    global counter
    for _ in range(200_000):
        counter += 1                # не атомарно -> втрачаємо оновлення

threads = [threading.Thread(target=increment) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print("БЕЗ Lock:", counter, "(очікували 800000 — але через гонку менше)")

In [ ]:
import threading

# ✅ З блокуванням — критична секція під Lock, результат правильний
counter = 0
lock = threading.Lock()
def increment_safe():
    global counter
    for _ in range(200_000):
        with lock:                  # тільки один потік у цьому блоці одночасно
            counter += 1

threads = [threading.Thread(target=increment_safe) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print("З Lock:   ", counter, "(рівно 800000)")

**Про блокування (питають на співбесіді):**
- **`Lock`** — базовий м'ютекс: `acquire()`/`release()` (краще через `with lock:`).
- **`RLock`** (reentrant) — той самий потік може захопити його кілька разів (для рекурсії).
- **Deadlock (взаємне блокування)** — два потоки чекають лока один одного назавжди. Уникають:
  завжди захоплювати локи в **однаковому порядку**, використовувати таймаути.

### `Semaphore` — обмежити кількість одночасних

**Семафор** дозволяє одночасно **не більше N** потокам увійти в секцію (напр. не більше 2
одночасних «завантажень»).

In [ ]:
import threading, time

sem = threading.Semaphore(2)        # максимум 2 одночасно

def download(name):
    with sem:                       # якщо вже 2 всередині — решта чекають
        print(f"  {name}: качаю...")
        time.sleep(0.3)
        print(f"  {name}: готово")

threads = [threading.Thread(target=download, args=(f"file-{i}",)) for i in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print("Готово (качалось по 2 одночасно)")

### `Queue` — безпечний обмін даними між потоками

`queue.Queue` — **потокобезпечна** черга (сама всередині має локи). Класичний патерн
**producer-consumer**: одні потоки кладуть завдання, інші забирають.

In [ ]:
import threading, queue, time

q = queue.Queue()

def producer():
    for i in range(5):
        q.put(i)                    # кладемо завдання
    q.put(None)                     # "стоп"-сигнал

def consumer():
    while True:
        item = q.get()
        if item is None: break
        print("  обробив:", item)
        q.task_done()

tp = threading.Thread(target=producer)
tc = threading.Thread(target=consumer)
tp.start(); tc.start()
tp.join(); tc.join()
print("Черга опрацьована")

### `ThreadPoolExecutor` — пул потоків (найзручніше)

Замість того щоб керувати потоками вручну, беруть **пул** із `concurrent.futures`. `map` розподіляє
роботу по потоках і збирає результати.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def fetch(n):
    time.sleep(0.2)                 # імітація I/O
    return n * n

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    results = list(pool.map(fetch, range(8)))
print("результати:", results)
print(f"8 задач по 0.2с у пулі з 4 потоків: {time.perf_counter() - start:.2f}с")

## 4. `multiprocessing` — процеси (для CPU-bound)

Щоб **паралелити обчислення** попри GIL, запускають **окремі процеси** — у кожного власний
інтерпретатор Python і **власний GIL**. Так задіюються всі ядра CPU.

> ⚠️ **Чому приклади нижче — як текст, а не запускні клітинки.** На macOS/Windows `multiprocessing`
> використовує метод старту **`spawn`**: дочірній процес **заново імпортує** ваш модуль. У Jupyter
> «модуль» — це сам ноутбук, тому функції з клітинок **не серіалізуються (pickling)** коректно, і
> код зависає/падає. Тому `multiprocessing` запускають **як окремий `.py`-файл** із захистом
> `if __name__ == "__main__":`. Це саме по собі — часте питання співбесіди.

### Приклад: пул процесів для CPU-bound задачі
```python
# файл cpu_task.py  — запускати як `python cpu_task.py`
from multiprocessing import Pool
import time

def heavy(n):                       # важке обчислення (CPU-bound)
    return sum(i * i for i in range(n))

if __name__ == "__main__":          # ОБОВ'ЯЗКОВО при spawn
    numbers = [10_000_000] * 8
    start = time.perf_counter()
    with Pool(processes=4) as pool: # 4 процеси на 4 ядрах
        results = pool.map(heavy, numbers)
    print(results[:1], f"час: {time.perf_counter() - start:.2f}с")
    # На 4 ядрах це у ~4 рази швидше, ніж послідовно чи через threading
```

### Те саме через `ProcessPoolExecutor` (єдиний інтерфейс із ThreadPoolExecutor)
```python
from concurrent.futures import ProcessPoolExecutor

if __name__ == "__main__":
    with ProcessPoolExecutor(max_workers=4) as pool:
        results = list(pool.map(heavy, [10_000_000] * 8))
```

### Обмін даними між процесами (IPC)

Процеси **не ділять пам'ять** (на відміну від потоків), тож дані треба **передавати явно**:
- **`Queue`** / **`Pipe`** — передати повідомлення між процесами;
- **`Value`** / **`Array`** — спільна пам'ять для простих типів;
- **`Manager`** — спільні `list`/`dict`;
- **`Lock`** — є і для процесів (`multiprocessing.Lock`).

```python
from multiprocessing import Process, Queue

def worker(q):
    q.put("привіт з процесу")

if __name__ == "__main__":
    q = Queue()
    p = Process(target=worker, args=(q,))
    p.start()
    print(q.get())              # отримуємо дані з іншого процесу
    p.join()
```

> 💡 Передача даних між процесами **дорожча** за потоки (серіалізація/pickling + копіювання).
> Тому `multiprocessing` вигідний, коли обчислення **важкі**, а даних передається небагато.

## 5. 🔎 `asyncio` — асинхронність (детально)

Найпотужніший інструмент для **I/O-bound із тисячами одночасних задач** (веб-сервери, скрапери,
чати). Головна ідея: **один потік**, але замість очікування задача **добровільно віддає керування**
на час I/O, і подія-цикл дає попрацювати іншим.

### 5.1. Event loop, корутини, `await`
- **Event loop (цикл подій)** — «диспетчер», що крутиться в одному потоці й **перемикає** задачі:
  запускає корутину, поки та не дійде до `await`, тоді ставить її «на паузу» й бере наступну.
- **Корутина** — функція, оголошена через **`async def`**. Її виклик **не виконує** тіло, а
  повертає об'єкт-корутину (як генератор — лінивий).
- **`await`** — «зачекати результат, віддавши керування циклу на цей час». Можна `await` лише те,
  що є awaitable (інша корутина, `Task`, `asyncio.sleep`…).

> **Ключова відмінність від потоків:** перемикання **кооперативне** — лише на `await` (а не будь-коли,
> як у threading). Тому race condition тут набагато рідші, а між `await` код виконується атомарно.

In [ ]:
import asyncio, time

async def fetch(name, delay):
    print(f"  {name}: старт")
    await asyncio.sleep(delay)       # імітація I/O; НЕ блокує потік (на відміну від time.sleep!)
    print(f"  {name}: готово за {delay}с")
    return name.upper()

# Послідовно vs конкурентно
async def main():
    start = time.perf_counter()
    # ❌ послідовно (await один за одним) -> сума часів
    await fetch("seq-A", 0.5)
    await fetch("seq-B", 0.5)
    print(f"послідовно: {time.perf_counter() - start:.1f}с\n")

    start = time.perf_counter()
    # ✅ конкурентно через gather -> час = МАКСИМУМ, а не сума
    results = await asyncio.gather(fetch("par-A", 0.5), fetch("par-B", 0.5), fetch("par-C", 0.5))
    print(f"конкурентно (gather): {time.perf_counter() - start:.1f}с, результати: {results}")

await main()      # у Jupyter; у звичайному .py -> asyncio.run(main())

> ⚠️ **`asyncio.sleep` vs `time.sleep`.** `await asyncio.sleep(1)` — «неблокуюче» очікування:
> звільняє цикл для інших задач. А `time.sleep(1)` **заблокує весь event loop** — і вся асинхронність
> зупиниться. У async-коді **ніколи** не викликайте блокуючі функції напряму (див. 5.4).

### 5.2. `Task` — запустити корутину «у фоні»

`asyncio.create_task(coro)` **одразу планує** корутину на виконання (не чекаючи `await`). Так
запускають кілька задач, а тоді збирають результати.

In [ ]:
import asyncio

async def work(n):
    await asyncio.sleep(0.3)
    return n * 2

async def main():
    t1 = asyncio.create_task(work(1))    # стартують у фоні одразу
    t2 = asyncio.create_task(work(2))
    print("задачі запущені, займаємось іншим...")
    print("результати:", await t1, await t2)   # тепер дочекаємось

await main()

### 5.3. `Lock` та `Semaphore` в asyncio

Хоча потік один, дані все одно можна «пошкодити», якщо між `await` два таски змінюють спільний
стан. Для цього є **`asyncio.Lock`** (критична секція) та **`asyncio.Semaphore`** (обмежити
кількість одночасних — напр. не бомбардувати сервер сотнями запитів).

In [ ]:
import asyncio

# asyncio.Lock: захищаємо спільний стан у критичній секції з await всередині
async def main():
    lock = asyncio.Lock()
    shared = {"v": 0}

    async def inc():
        async with lock:                 # лише один таск усередині
            tmp = shared["v"]
            await asyncio.sleep(0)        # віддаємо керування -> без локу була б гонка
            shared["v"] = tmp + 1

    await asyncio.gather(*(inc() for _ in range(1000)))
    print("asyncio.Lock -> результат:", shared["v"], "(рівно 1000)")

await main()

In [ ]:
import asyncio

# asyncio.Semaphore: не більше 2 "завантажень" одночасно
async def download(sem, name):
    async with sem:
        print(f"  {name}: качаю")
        await asyncio.sleep(0.4)
        print(f"  {name}: готово")

async def main():
    sem = asyncio.Semaphore(2)           # ліміт конкурентності = 2
    await asyncio.gather(*(download(sem, f"file-{i}") for i in range(5)))

await main()

### 5.4. Блокуючий / CPU-код в asyncio: `to_thread` та `run_in_executor`

Якщо треба викликати **блокуючу** функцію (стара бібліотека, важке обчислення) — не робіть це
напряму (заблокуєте цикл). Винесіть її в **окремий потік** (`asyncio.to_thread`) або пул через
`loop.run_in_executor`. Для **CPU-bound** — у **ProcessPoolExecutor** (обхід GIL).

In [ ]:
import asyncio, time

def blocking_io(n):                  # звичайна БЛОКУЮЧА функція
    time.sleep(0.3)
    return n * n

async def main():
    # to_thread переносить блокуючий виклик у потік, щоб не заморозити event loop
    result = await asyncio.to_thread(blocking_io, 5)
    print("результат із потоку:", result)

await main()

> 🔑 **Коли asyncio НЕ підходить:** для **CPU-bound** він не дасть прискорення (один потік, GIL).
> Тоді — `multiprocessing`. asyncio блищить саме на **I/O з високою конкурентністю**.

## 6. 🔎 Порівняння й вибір (як на співбесіді)

| | `threading` | `multiprocessing` | `asyncio` |
|---|---|---|---|
| Модель | кілька потоків, спільна пам'ять | кілька процесів, окрема пам'ять | один потік, event loop |
| Паралелізм CPU | ❌ (GIL) | ✅ (свій GIL у процесі) | ❌ |
| Для чого | **I/O-bound** | **CPU-bound** | **I/O-bound, тисячі задач** |
| Перемикання | будь-коли (преемптивне) | процеси незалежні | лише на `await` (кооперативне) |
| Обмін даними | спільна пам'ять (+локи) | IPC / serialization (дорого) | спільна пам'ять (один потік) |
| Ціна | помірна | висока (процеси, pickling) | низька (легкі корутини) |
| Race conditions | часто (потрібні локи) | рідше (окрема пам'ять) | рідко (лише навколо `await`) |

### Шпаргалка вибору
- Чекаєте **мережу/диск/БД**, задач **небагато** → `threading` (простіше) або `asyncio`.
- Чекаєте I/O, задач **тисячі** (веб-сервер, скрапер) → **`asyncio`**.
- **Важкі обчислення** (CPU) → **`multiprocessing`**.
- Треба і те, й те → комбінують: `asyncio` + `ProcessPoolExecutor` через `run_in_executor`.

> 🎤 **Коротка відповідь на співбесіді:** «Потоки й asyncio — для I/O-bound (GIL не заважає, бо під
> час очікування він відпускається), причому asyncio масштабується краще на тисячі задач завдяки
> легким корутинам і кооперативному перемиканню. Для CPU-bound потрібні процеси —
> `multiprocessing`, бо в кожного власний GIL і задіюються всі ядра.»

# ✅ Підсумок

- **Concurrency** (перемикання задач) ≠ **parallelism** (одночасно на ядрах).
- **GIL:** у CPython байткод виконує 1 потік; тому потоки не пришвидшують **CPU**, але допомагають на **I/O**.
- **`threading`** — I/O-bound; спільна пам'ять → потрібні **`Lock`/`RLock`**, **`Semaphore`** (ліміт),
  **`Queue`** (обмін), **`ThreadPoolExecutor`** (пул).
- **`multiprocessing`** — CPU-bound; окремі процеси зі своїм GIL; IPC (`Queue`/`Pipe`/`Value`/`Manager`);
  запускати як `.py` із `if __name__ == "__main__":` (spawn/pickling).
- **`asyncio`** — I/O з високою конкурентністю; **event loop**, `async`/`await`, `create_task`,
  `gather`; **`Lock`/`Semaphore`/`Queue`**; блокуючий код — через `to_thread`/`run_in_executor`.
- Вибір: I/O мало → threading; I/O багато → asyncio; CPU → multiprocessing.

### 📚 Хочу знати більше
- `asyncio` (офіційно): <https://docs.python.org/3/library/asyncio.html>
- Real Python — Async IO: <https://realpython.com/async-io-python/>
- Real Python — Threading: <https://realpython.com/intro-to-python-threading/>
- Real Python — Multiprocessing: <https://realpython.com/python-concurrency/>
- Про GIL: <https://realpython.com/python-gil/>